# Inspect test-run output — CAM & CLM variable tables

Tabulates the variables actually written by the production `h0` tapes (one CAM test, one CLM test),
and checks the **requested whitelist vs what actually landed** in the file.

Outputs per model: `vars_<model>.csv` and a printed present / missing / unexpected report.


## 1 · Configuration — point at your two test files

In [5]:
from pathlib import Path
import glob

dir_path = "/cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics"
CAM_GLOB = dir_path + "/atm/hist/*cam.h0*.nc"
CLM_GLOB = dir_path + "/lnd/hist/*clm2.h0*.nc"

OUT_DIR = Path("check_output"); OUT_DIR.mkdir(exist_ok=True)
RECORDS_PER_MEMBER = 120   # 10 yr monthly, for size projection

def first_match(pat):
    hits = sorted(glob.glob(pat, recursive=True))
    if not hits: print(f"[!] no file matched {pat}"); return None
    if len(hits) > 1: print(f"[i] {len(hits)} matched {pat}; using {hits[0]}")
    return hits[0]

CAM_FILE = first_match(CAM_GLOB)
CLM_FILE = first_match(CLM_GLOB)
print("CAM:", CAM_FILE); print("CLM:", CLM_FILE)


[i] 2 matched /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/atm/hist/*cam.h0*.nc; using /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/atm/hist/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics.cam.h0.2000-01-01-00000.nc
[i] 2 matched /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/lnd/hist/*clm2.h0*.nc; using /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/lnd/hist/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics.clm2.h0.2000-01-01-00000.nc
CAM: /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/atm/hist/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics.cam.h0.2000-01-01-00000.nc
CLM: /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/l

## 2 · Functions (metadata-only) + the whitelist branch maps

In [6]:
import numpy as np, pandas as pd, xarray as xr

# dims that are NOT an extra (vertical / pft / column) level:
FLAT_DIMS = {"time","lat","lon","lonu","latu","nbnd","hist_interval","chars","string_length"}

def build_var_table(path):
    """One row per data variable. Lazy open, metadata only — no data is read."""
    ds = xr.open_dataset(path, decode_times=False, decode_cf=False)
    rows=[]
    for name, da in ds.data_vars.items():
        dims=tuple(da.dims); sizes={d:ds.sizes[d] for d in dims}
        extra=[d for d in dims if d not in FLAT_DIMS]           # vertical/pft/etc.
        nontime=[sizes[d] for d in dims if d!="time"]
        per=(int(np.prod(nontime)) if nontime else 1)*da.dtype.itemsize
        rows.append(dict(variable=name, dims=",".join(dims),
            multilevel=bool(extra), level_dim=",".join(extra),
            dtype=str(da.dtype), shape=",".join(str(sizes[d]) for d in dims),
            mib_per_rec=per/1024**2,
            units=da.attrs.get("units",""), long_name=da.attrs.get("long_name","")))
    ds.close()
    return pd.DataFrame(rows).sort_values("variable").reset_index(drop=True)

def tag_and_report(df, branches, model, records=RECORDS_PER_MEMBER):
    """Add a 'branch' column, write CSV, print counts + present/missing/unexpected."""
    v2b = {v:b for b,vs in branches.items() for v in vs}
    df = df.copy()
    df["branch"] = df.variable.map(v2b).fillna("(unexpected)")
    df.to_csv(OUT_DIR/f"vars_{model}.csv", index=False)

    gib = lambda s: s*records/1024
    present = set(df.variable)
    requested = set(v2b)
    missing = sorted(requested - present)          # asked for, not in file
    unexpected = sorted(df.loc[df.branch=="(unexpected)","variable"])  # in file, not requested

    print(f"===== {model.upper()} : {CAM_FILE if model=='cam' else CLM_FILE} =====")
    print(f"variables in file : {len(df)}  (multilevel={int(df.multilevel.sum())}, "
          f"2D={int((~df.multilevel).sum())})")
    print(f"projected size    : ~{gib(df.mib_per_rec.sum()):.2f} GiB/member (uncompressed, {records} recs)")
    print("\nby branch (count | GiB/member):")
    g=(df.assign(g=lambda d:gib(d.mib_per_rec)).groupby("branch")
         .agg(n=("variable","size"), gib=("g","sum")).sort_values("gib",ascending=False))
    print(g.to_string())
    print(f"\nMISSING (requested but absent): {len(missing)}")
    if missing: print("  ", missing)
    print(f"UNEXPECTED (in file, not requested): {len(unexpected)}")
    if unexpected: print("  ", unexpected)
    return df

# ---- whitelist branch maps (from production_diagnostics / clm_diagnostics) ----
CAM_BRANCHES = {
 "radiative": "FSNT FSNTC FLNT FLNTC FLUT FLUTC FSNTOA FSNTOAC SOLIN FSNS FSNSC FLNS FLNSC FSDS FSDSC FLDS SWCF LWCF".split(),
 "ghan_drf": "FSNT_DRF FLNT_DRF FSNTCDRF FLNTCDRF FSDS_DRF FSDSCDRF FSUTADRF FSUS_DRF FLUS".split(),
 "bvoc": "SFISOP SFMTERP SFBCARY cb_ISOP cb_MTERP cb_BCARY MEG_ISOP MEG_MTERP MEG_BCARY emis_ISOP emis_MTERP ISOP MTERP BCARY".split(),
 "soa_core": "SOA_LV SOA_SV H2SO4 SOA_NA SOA_A1 SO4_NA SO4_A1 N_AER cb_SOA_LV cb_SOA_SV cb_H2SO4".split(),
 "npf": "NUCLRATE FORMRATE COAGNUCL GR GRH2SO4 GRSOA ORGNUCL NUCLSOA".split(),
 "tendencies": "SOA_NAcondTend SOA_A1condTend SOA_NAcoagTend SOA_A1coagTend SOA_NA_mixnuc1 SOA_A1_mixnuc1 SO4_NAcondTend SO4_A1condTend SO4_NAcoagTend SO4_A1coagTend SO4_NA_mixnuc1 SO4_A1_mixnuc1".split(),
 "deposition": "SOA_NADDF SOA_A1DDF SO4_NADDF SO4_A1DDF DF_H2SO4 SOA_NASFWET SOA_A1SFWET SO4_NASFWET SO4_A1SFWET WD_A_H2SO4 WD_H2SO4".split(),
 "ccn": "CCN1 CCN2 CCN3 CCN4 CCN5 CCN6 CCN7 CCN_B".split(),
 "optics": "AOD_VIS AEROD_v DOD550 DOD440 DOD870 ABS550 ABS550_A OD550DRY AB550DRY CABS550 A550_BC A550_POM A550_SO4 A550_SS A550_DU".split(),
 "cloud": "CDNUMC TGCLDLWP TGCLDIWP TGCLDCWP CLDTOT CLDLOW CLDMED CLDHGH ACTREL ACTREI ACTNL FCTL FCTI CLOUD CLDLIQ CLDICE AREL AREI AWNC FREQL FREQI NUMLIQ NUMICE".split(),
 "biogeophysical": "TS TREFHT SHFLX LHFLX PRECC PRECL PRECSC PRECSL TAUX TAUY PSL U10 QREFHT LANDFRAC OCNFRAC ICEFRAC SNOWHLND".split(),
 "ch4_ozone": "O3 OH CH4 NO NO2 CO HO2 TROP_P TROP_T TROP_Z".split(),
 "meteorology": "T Q U V OMEGA Z3 PS".split(),
}
CLM_BRANCHES = {
 "radiation_albedo": "FSA FSR FIRA FIRE FSH EFLX_LH_TOT FGR FSDS FLDS FSDSVD FSDSVI FSDSND FSDSNI FSRVD FSRND".split(),
 "snow": "H2OSNO SNOWDP FSNO SNOWLIQ SNOWICE".split(),
 "temperature": "TSA TV TG TSKIN TSOI".split(),
 "vegetation": "TLAI ELAI LAISUN LAISHA TSAI HTOP".split(),
 "megan_drivers": "PARVEGLN BTRANMN".split(),
 "hydrology_et": "QFLX_EVAP_TOT QSOIL QVEGE QVEGT QINTR QOVER QRUNOFF RAIN SNOW H2OSOI SOILLIQ SOILICE ZWT".split(),
 "carbon": "GPP NPP AR HR NEE".split(),
 "other": "WIND PCT_NAT_PFT".split(),
 "megan_emissions": ("MEG_isoprene MEG_carene_3 MEG_limonene MEG_myrcene MEG_pinene_a MEG_pinene_b "
                     "MEG_acetaldehyde MEG_acetic_acid MEG_acetone MEG_ethanol MEG_formaldehyde MEG_methanol").split(),
 "megan_gamma": ("GAMMAL GAMMAS GAMMAC_isoprene "
                 "EPS_isoprene GAMMA_isoprene GAMMAP_isoprene GAMMAT_isoprene GAMMAA_isoprene "
                 "EPS_pinene_a GAMMA_pinene_a GAMMAP_pinene_a GAMMAT_pinene_a GAMMAA_pinene_a "
                 "EPS_carene_3 GAMMA_carene_3 GAMMAP_carene_3 GAMMAT_carene_3 GAMMAA_carene_3 "
                 "EPS_pinene_b GAMMA_pinene_b GAMMAP_pinene_b GAMMAT_pinene_b GAMMAA_pinene_b "
                 "EPS_myrcene GAMMA_myrcene GAMMAP_myrcene GAMMAT_myrcene GAMMAA_myrcene "
                 "EPS_limonene GAMMA_limonene GAMMAP_limonene GAMMAT_limonene GAMMAA_limonene").split(),
}


## 3 · CAM test file

In [7]:
cam = tag_and_report(build_var_table(CAM_FILE), CAM_BRANCHES, "cam") if CAM_FILE else None
cam.head(20) if cam is not None else None


===== CAM : /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/atm/hist/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics.cam.h0.2000-01-01-00000.nc =====
variables in file : 188  (multilevel=54, 2D=134)
projected size    : ~10.59 GiB/member (uncompressed, 120 recs)

by branch (count | GiB/member):
                 n       gib
branch                      
cloud           23  2.057877
soa_core        11  1.600571
npf              8  1.582031
ch4_ozone       10  1.402817
ccn              8  1.390457
meteorology      7  1.192703
bvoc            14  0.661240
optics          15  0.284271
radiative       18  0.111237
biogeophysical  17  0.105057
tendencies      12  0.074158
deposition      11  0.067978
ghan_drf         9  0.055618
(unexpected)    25  0.000216

MISSING (requested but absent): 0
UNEXPECTED (in file, not requested): 25
   ['P0', 'ch4vmr', 'co2vmr', 'date', 'date_written', 'datesec', 'f11vmr', 'f12vmr', 'gw', 'hyai'

,variable,dims,multilevel,level_dim,dtype,shape,mib_per_rec,units,long_name,branch
0,A550_BC,"time,lat,lon",False,,float32,"1,96,144",0.052734,unitless,BC abs. aerosol optical depth 550nm,optics
1,A550_DU,"time,lat,lon",False,,float32,"1,96,144",0.052734,unitless,mineral abs. aerosol optical depth 550nm,optics
2,A550_POM,"time,lat,lon",False,,float32,"1,96,144",0.052734,unitless,OC abs. aerosol optical depth 550nm,optics
3,A550_SO4,"time,lat,lon",False,,float32,"1,96,144",0.052734,unitless,SO4 aerosol abs. optical depth 550nm,optics
4,A550_SS,"time,lat,lon",False,,float32,"1,96,144",0.052734,unitless,sea-salt abs aerosol optical depth 550nm,optics
5,AB550DRY,"time,lat,lon",False,,float32,"1,96,144",0.052734,unitless,Dry aerosol absorptive optical depth at 550nm,optics
6,ABS550,"time,lat,lon",False,,float32,"1,96,144",0.052734,unitless,Aerosol absorptive optical depth at 550nm,optics
7,ABS550_A,"time,lev,lat,lon",True,lev,float32,"1,32,96,144",1.687500,m-1,aerosol absorption coefficient,optics
8,ACTNL,"time,lat,lon",False,,float32,"1,96,144",0.052734,m-3,Average Cloud Top droplet number,cloud
9,ACTREI,"time,lat,lon",False,,float32,"1,96,144",0.052734,Micron,Average Cloud Top ice effective radius,cloud


## 4 · CLM test file

In [8]:
clm = tag_and_report(build_var_table(CLM_FILE), CLM_BRANCHES, "clm") if CLM_FILE else None
clm.head(20) if clm is not None else None


===== CLM : /cluster/work/users/adelez/archive/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics/lnd/hist/NF2000norbc_tropstratchem_nudg_ctrl_f19_f19-test-new-diagnostics.clm2.h0.2000-01-01-00000.nc =====
variables in file : 555  (multilevel=43, 2D=512)
projected size    : ~17.48 GiB/member (uncompressed, 120 recs)

by branch (count | GiB/member):
                    n        gib
branch                          
(unexpected)      457  15.097282
hydrology_et       13   0.865173
megan_gamma        33   0.407867
temperature         5   0.358429
other               2   0.197754
radiation_albedo   15   0.185394
megan_emissions    12   0.148315
vegetation          6   0.074158
carbon              5   0.061798
snow                5   0.061798
megan_drivers       2   0.024719

MISSING (requested but absent): 0
UNEXPECTED (in file, not requested): 457
   ['ACTUAL_IMMOB', 'AGNPP', 'ALT', 'ALTMAX', 'ATM_TOPO', 'BAF_CROP', 'BAF_PEATF', 'BCDEP', 'BGNPP', 'BSW', 'BTRAN2', 'CH4PROD', '

,variable,dims,multilevel,level_dim,dtype,shape,mib_per_rec,units,long_name,branch
0,ACTUAL_IMMOB,"time,lat,lon",False,,float64,"1,96,144",0.105469,gN/m^2/s,actual N immobilization,(unexpected)
1,AGNPP,"time,lat,lon",False,,float64,"1,96,144",0.105469,gC/m^2/s,aboveground NPP,(unexpected)
2,ALT,"time,lat,lon",False,,float64,"1,96,144",0.105469,m,current active layer thickness,(unexpected)
3,ALTMAX,"time,lat,lon",False,,float64,"1,96,144",0.105469,m,maximum annual active layer thickness,(unexpected)
4,AR,"time,lat,lon",False,,float64,"1,96,144",0.105469,gC/m^2/s,autotrophic respiration (MR + GR),carbon
5,ATM_TOPO,"time,lat,lon",False,,float64,"1,96,144",0.105469,m,atmospheric surface height,(unexpected)
6,BAF_CROP,"time,lat,lon",False,,float64,"1,96,144",0.105469,proportion/sec,fractional area burned for crop,(unexpected)
7,BAF_PEATF,"time,lat,lon",False,,float64,"1,96,144",0.105469,proportion/sec,fractional area burned in peatland,(unexpected)
8,BCDEP,"time,lat,lon",False,,float64,"1,96,144",0.105469,kg/m^2/s,total BC deposition (dry+wet) from atmosphere,(unexpected)
9,BGNPP,"time,lat,lon",False,,float64,"1,96,144",0.105469,gC/m^2/s,belowground NPP,(unexpected)


## 5 · How to read the report

- **MISSING** = you requested it in `fincl` but it's not in the file → silently dropped (wrong name) or gated by a flag that's off. Investigate before trusting the run.
- **UNEXPECTED** = in the file but not in your whitelist → usually an automatic coordinate/bounds/scalar (`hyam`, `time_bnds`, `date`…), which is fine. Anything else means the tape wasn't as clean as intended (check `empty_htapes`).
- **multilevel** flags vertical/pft/column fields (CAM `lev/ilev`; CLM `levsoi/levgrnd/levsno/natpft`) — the size drivers.
